In [2]:
# Retail & Marketing Analytics Project
# Notebook 4: KPI Calculation


# ============================================================================
# 1. IMPORT LIBRARIES AND LOAD DATA
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load all processed data
df_sales = pd.read_csv('data/processed/cleaned_retail_sales.csv')
df_sales['Order_Date'] = pd.to_datetime(df_sales['Order_Date'])

rfm = pd.read_csv('data/processed/rfm_analysis.csv')
clv = pd.read_csv('data/processed/customer_clv.csv')

print("="*80)
print("KPI CALCULATION")
print("="*80)

# ============================================================================
# 2. CREATING KPIs
# ============================================================================

print("\n" + "="*80)
print("CREATING KPIs")
print("="*80)

kpis = {}

# Business Metrics
kpis['Total_Revenue'] = df_sales['Sales'].sum()
kpis['Total_Orders'] = df_sales['Order_ID'].nunique()
kpis['Total_Customers'] = df_sales['Customer_ID'].nunique()

kpis['Avg_Order_Value'] = (df_sales.groupby('Order_ID')['Sales'].sum().mean())

kpis['Revenue_Per_Customer'] = (kpis['Total_Revenue'] / kpis['Total_Customers'])

# Customer Metrics
customer_orders = df_sales.groupby('Customer_ID')['Order_ID'].nunique()

kpis['Repeat_Customers'] = (customer_orders > 1).sum()
kpis['Repeat_Customer_Rate'] = ( kpis['Repeat_Customers'] / kpis['Total_Customers'] * 100 )

# RFM Segment Summary
segment_summary = rfm.groupby('Segment').agg({
    'Customer_ID': 'count',
    'Monetary':'sum',
    'Frequency':'mean',
    'Recency':'mean'
}).reset_index()

# Simple CLV
avg_clv = clv['CLV'].mean()

# Monthly KPI Trends
df_sales['YearMonth'] = df_sales['Order_Date'].dt.to_period('M')

monthly_kpis = df_sales.groupby('YearMonth').agg({
    'Sales':'sum',
    'Order_ID':'nunique',
    'Customer_ID':'nunique'
}).reset_index()

monthly_kpis['AOV'] = (monthly_kpis['Sales'] / monthly_kpis['Order_ID'])

monthly_kpis['YearMonth'] = monthly_kpis['YearMonth'].astype(str)

# Exporting data to csv

kpi_df = pd.DataFrame(list(kpis.items()), columns=['KPI','Value'])
kpi_df.to_csv('outputs/reports/kpi_summary.csv', index=False)

segment_summary.to_csv('outputs/reports/rfm_segment_summary.csv', index=False)
monthly_kpis.to_csv('outputs/reports/monthly_kpis.csv', index=False)

print("Saved Data to outputs/reports")




KPI DESIGN 

CREATING KPIs
Saved Data to outputs/reports


In [ ]:
def format_value(value):
    if value >=1_000_000:
        return f"{value / 1_000_000:.2f}M"
    elif value >=1_000:
        return f"{value / 1_000:.2f}k"
    else:
        return str(value)

kpi_df['Value'] = kpi_df['Value'].apply(format_value)
kpi_df